In [1]:

import json, os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve
from sklearn.ensemble import GradientBoostingClassifier

# ---- config ----
EVENTS_FILE = "events_England.json"
OUTDIR = Path("outputs")
MIN_PASSES = 150
RNG = 42

# deterministic figures
plt.rcParams.update({
    "savefig.dpi": 300,
    "figure.dpi": 100,
    "figure.figsize": (7.0, 5.0)
})

# ---------------- IO ----------------
def _read_json_any(path):
    text = Path(path).read_text(encoding="utf-8").lstrip("\ufeff")
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # NDJSON fallback
        return [json.loads(line) for line in text.splitlines() if line.strip()]

def load_events_dataframe(path):
    obj = _read_json_any(path)
    rows = []
    if isinstance(obj, dict) and "events_per_match" in obj:
        matches = obj.get("matches", [])
        for i, evs in enumerate(obj["events_per_match"]):
            mid = matches[i] if i < len(matches) else None
            for e in evs:
                d = dict(e)
                if "match_id" not in d and mid is not None:
                    d["match_id"] = mid
                rows.append(d)
    elif isinstance(obj, list):
        stack = list(obj)
        while stack:
            x = stack.pop()
            if isinstance(x, dict):
                rows.append(x)
            elif isinstance(x, list):
                stack.extend(x)
    else:
        raise ValueError("Unsupported JSON structure in events_England.json")
    if not rows:
        raise ValueError("No events found in events_England.json")

    df = pd.json_normalize(rows, sep="_")
    df["minute"] = pd.to_numeric(df.get("minute"), errors="coerce").fillna(0).astype(int)
    df["second"] = pd.to_numeric(df.get("second"), errors="coerce").fillna(0.0)
    df["time_s"]  = df["minute"] * 60 + df["second"]
    return df

# ---------------- features ----------------
def build_pass_table(events):
    if "pass_outcome_name" not in events:
        events["pass_outcome_name"] = np.nan
    p = events[(events.get("type_name") == "Pass") & (events["pass_outcome_name"].isna())].copy()

    def comp(v, i): return v[i] if isinstance(v, list) and len(v) >= 2 else np.nan
    p["x"] = p.get("location").apply(lambda v: comp(v, 0))
    p["y"] = p.get("location").apply(lambda v: comp(v, 1))
    if "pass_end_location" not in p: p["pass_end_location"] = np.nan
    p["end_x"] = p["pass_end_location"].apply(lambda v: comp(v, 0))
    p["end_y"] = p["pass_end_location"].apply(lambda v: comp(v, 1))

    p = p.dropna(subset=["x", "y", "end_x", "end_y"]).copy()

    dx = p["end_x"] - p["x"]
    dy = p["end_y"] - p["y"]
    p["dist"]    = np.hypot(dx, dy)
    p["angle"]   = np.degrees(np.arctan2(dy, dx))
    p["forward"] = (dx > 0).astype(int)

    for col in ["pass_height_name", "pass_body_part_name"]:
        if col not in p: p[col] = ""
        p[col] = p[col].fillna("Unknown")

    # keys and safe labels
    for col in ["match_id", "possession", "team_id", "team_name", "player_name"]:
        if col not in p: p[col] = np.nan
    p["player_name"] = p["player_name"].fillna("Unknown")
    p["team_name"]   = p["team_name"].fillna("Unknown")
    return p

# ---------------- labeling (safe indexing) ----------------
def label_shot_within_10s(passes, events, horizon=10):
    shots = events[events.get("type_name") == "Shot"][["match_id", "possession", "team_id", "time_s"]].copy()
    shots = shots.dropna(subset=["time_s"]).sort_values(["match_id","possession","team_id","time_s"])

    key = ["match_id", "possession", "team_id"]
    passes = passes.sort_values(key + ["time_s"]).reset_index(drop=True)
    label = np.zeros(len(passes), dtype=np.int8)

    for gk, gp in passes.groupby(key, sort=False):
        gs = shots[
            (shots["match_id"] == gk[0]) &
            (shots["possession"] == gk[1]) &
            (shots["team_id"] == gk[2])
        ]
        if gs.empty:
            continue
        pt = gp["time_s"].to_numpy()
        st = gs["time_s"].to_numpy()  # sorted
        idx = np.searchsorted(st, pt, side="left")

        dt = np.full(pt.shape, np.inf, dtype=float)
        valid = idx < st.shape[0]
        dt[valid] = st[idx[valid]] - pt[valid]

        ok = (dt >= 0) & (dt <= horizon)
        label[gp.index.values] = ok.astype(np.int8)

    passes["label_shot10"] = label
    return passes

# ---------------- model ----------------
def train_eval(df):
    num = ["x", "y", "end_x", "end_y", "dist", "angle", "forward"]
    cat = ["pass_height_name", "pass_body_part_name"]
    X = df[num + cat]
    y = df["label_shot10"].astype(int).to_numpy()

    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RNG)

    pre = ColumnTransformer([
        ("num", "passthrough", num),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), cat),
    ])
    clf = GradientBoostingClassifier(n_estimators=400, learning_rate=0.05, max_depth=3, random_state=RNG)
    pipe = Pipeline([("pre", pre), ("clf", clf)]).fit(Xtr, ytr)

    proba = pipe.predict_proba(Xte)[:, 1]
    roc = roc_auc_score(yte, proba)
    ap  = average_precision_score(yte, proba)
    base = float(yte.mean())

    print("Model Performance:")
    print(f"  ROC-AUC: {roc:.3f}")
    print(f"  Average Precision: {ap:.3f}  (baseline={base:.3f})")
    print(f"  Test size: {len(yte)} | Positives in test: {int(yte.sum())}")

    prec, rec, _ = precision_recall_curve(yte, proba)
    OUTDIR.mkdir(exist_ok=True)
    plt.figure()
    plt.plot(rec, prec, label=f"GBoost (AP={ap:.3f})")
    plt.hlines(base, 0, 1, linestyles="dashed", label=f"baseline={base:.3f}")
    plt.xlabel("Recall"); plt.ylabel("Precision")
    plt.title("PL test — Precision–Recall (GBoost)")
    plt.legend(loc="lower left"); plt.tight_layout()
    plt.savefig(OUTDIR / "pl_pr_curve_GBoost.png", bbox_inches="tight")
    plt.close()
    return pipe

# ---------------- ranking ----------------
def export_player_leaders(passes, model):
    num = ["x", "y", "end_x", "end_y", "dist", "angle", "forward"]
    cat = ["pass_height_name", "pass_body_part_name"]
    feats = passes[num + cat]
    passes = passes.copy()
    passes["pass_danger"] = model.predict_proba(feats)[:, 1]

    agg = (passes.groupby(["player_name", "team_name"])["pass_danger"]
             .agg(passes="count", pass_danger="mean")
             .sort_values("pass_danger", ascending=False))
    OUTDIR.mkdir(exist_ok=True)
    agg.to_csv(OUTDIR / "pl_player_pass_danger_GBoost.csv")

    top = agg[agg["passes"] >= MIN_PASSES].head(10)
    plt.figure(figsize=(8.5, 5.0))
    if top.empty:
        plt.title(f"Premier League — no eligible players (≥{MIN_PASSES} passes)")
    else:
        labels = [f"{p} ({t})" for p, t in top.index]
        vals = top["pass_danger"].values
        plt.barh(labels[::-1], vals[::-1])
        for i, v in enumerate(vals[::-1]):
            plt.text(v, i, f"{v:.3f}", va="center", ha="left")
        plt.xlabel("mean predicted probability")
        plt.title(f"Premier League — top-10 by pass_danger (GBoost, ≥{MIN_PASSES} passes)")
    plt.tight_layout()
    plt.savefig(OUTDIR / "pl_top10_pass_danger_GBoost_min150.png", bbox_inches="tight")
    plt.close()

# ---------------- main ----------------
def main():
    if not Path(EVENTS_FILE).exists():
        raise FileNotFoundError(f"Missing {EVENTS_FILE} in current directory.")

    print("Loading events...")
    ev = load_events_dataframe(EVENTS_FILE)

    print("Building passes...")
    passes = build_pass_table(ev)

    print("Labeling (shot within 10s)...")
    passes = label_shot_within_10s(passes, ev, horizon=10)
    base = passes["label_shot10"].mean()
    pos  = int(passes["label_shot10"].sum())
    print(f"Passes with coords: {len(passes):,} | Positives: {pos:,} | Base rate: {base:.4f}")

    if passes["label_shot10"].nunique() < 2:
        print("Single-class labels. Writing baseline outputs and exiting.")
        OUTDIR.mkdir(exist_ok=True)
        plt.figure()
        plt.hlines(base, 0, 1, linestyles="dashed", label=f"baseline={base:.3f}")
        plt.xlabel("Recall"); plt.ylabel("Precision")
        plt.title("PL test — Precision–Recall (baseline only)")
        plt.legend(loc="lower left"); plt.tight_layout()
        plt.savefig(OUTDIR / "pl_pr_curve_GBoost.png", bbox_inches="tight"); plt.close()

        (passes.groupby(["player_name","team_name"])["label_shot10"]
           .agg(passes="count", pass_danger="mean")
           .rename(columns={"label_shot10":"pass_danger"})
           .to_csv(OUTDIR / "pl_player_pass_danger_GBoost.csv"))
        plt.figure(figsize=(8.5, 5.0))
        plt.title(f"Premier League — no eligible players (≥{MIN_PASSES} passes)")
        plt.tight_layout()
        plt.savefig(OUTDIR / "pl_top10_pass_danger_GBoost_min150.png", bbox_inches="tight")
        plt.close()
        return

    print("Training Gradient Boosting...")
    model = train_eval(passes)

    print("Exporting leaders...")
    export_player_leaders(passes, model)

    print("Done. Outputs written to:", OUTDIR.resolve())

if __name__ == "__main__":
    np.random.seed(RNG)
    os.environ["PYTHONHASHSEED"] = str(RNG)
    main()


Loading events...
Building passes...
Labeling (shot within 10s)...
Passes with coords: 8,787 | Positives: 576 | Base rate: 0.0656
Training Gradient Boosting...
Model Performance:
  ROC-AUC: 0.872
  Average Precision: 0.390  (baseline=0.065)
  Test size: 1758 | Positives in test: 115
Exporting leaders...
Done. Outputs written to: C:\Users\Rafallex\Documents\Uppsala\Semester 1\Mathematical modelling of football\A2\outputs
